# Day 04 — Data profiling: M5 + Amazon Reviews 2023

Profiling the two real datasets used later in the roadmap:
- **Walmart M5 Forecasting** → hourly staffing needs for ShiftWise
- **Amazon Reviews 2023 (Appliances)** → the review index for ShelfSense

Goal: understand each dataset's shape, types, missing values and memory use, then save clean Parquet copies.

**Sections**
1. Pandas basics on M5 *(this part)*
2. Memory profiling and dtype optimization
3. Reshaping wide → long and joining calendar + prices
4. Amazon reviews: reading and cleaning nested JSONL
5. Parquet vs CSV

## Part 1 — Pandas basics on M5

M5 has three files:
- `sales_train_evaluation.csv`: one row per item per store, one column per day (`d_1` … `d_1941`) with units sold
- `calendar.csv`: one row per day, with date, week and events
- `sell_prices.csv`: weekly price per item per store

Raw data lives in `data/raw/m5/` at the repo root and is not committed to Git.

### Setup
Find the repo root so paths work whether the notebook runs from the repo root or from `days/day04/`.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
M5 = RAW / "m5"
AMAZON = RAW / "amazon"
PROCESSED = ROOT / "data" / "processed"      # Parquet output goes here (Part 5)

print("M5 folder:", M5, "| exists:", M5.exists())
print("Amazon folder:", AMAZON, "| exists:", AMAZON.exists())
print("pandas version:", pd.__version__)

### Q1. Load the data — how big is each file, and how is text stored?
**Why:** First check on any dataset: rows, columns, and how Pandas typed the text columns.

In [ ]:
sales = pd.read_csv(M5 / "sales_train_evaluation.csv")
cal = pd.read_csv(M5 / "calendar.csv")
price = pd.read_csv(M5 / "sell_prices.csv")

print("sales:", sales.shape)
print("calendar:", cal.shape)
print("prices:", price.shape)
print("store_id dtype:", sales["store_id"].dtype)

sales.iloc[:3, :8]

### Q2. Column types and memory — where does the memory go?
**Why:** `info()` shows how many columns of each type there are and the rough memory use. This tells us what to optimize in Part 2.

In [ ]:
sales.info()

### Q3. Missing values — which columns have gaps, and what do the gaps mean?
**Why:** A missing value can mean *unknown* (the data exists but wasn't recorded) or *absence* (there is nothing to record). They are handled differently.

In [ ]:
print("sales missing values:", sales.isna().sum().sum())
print("prices missing values:", price.isna().sum().sum())
print()
cal_missing = cal.isna().sum()
print("calendar missing values per column:")
print(cal_missing[cal_missing > 0])
print()
print("days in calendar:", len(cal))
print("days with at least one event:", cal["event_name_1"].notna().sum())
print("days with two events:", cal["event_name_2"].notna().sum())

> In `calendar`, a missing `event_name_1` means **no event that day**, not unknown data. Absence like this usually becomes a real value (`"none"`) or a 0/1 flag when building features.

### Q4. Catalog size — how many items per category and per store?
**Why:** Shows how the data is distributed and whether every store carries the same catalog.

In [ ]:
print(sales["cat_id"].value_counts())
print()
print(sales["store_id"].value_counts())

### Q5. Filtering — how many CA food items, and what happens to the row labels?
**Why:** Filtering keeps the original index labels. That's why `loc` (labels) and `iloc` (positions) can give different results after a filter.

In [ ]:
mask = (sales["state_id"] == "CA") & (sales["cat_id"] == "FOODS")
ca_food = sales[mask]

print("CA + FOODS rows:", len(ca_food))
print("first 5 index labels:", ca_food.index[:5].tolist())
print("first row by position (iloc[0]):", ca_food.iloc[0]["id"])

### Q6. Grouping — which state sold the most on day 1?
**Why:** `groupby` is split → apply → combine, the same idea as SQL `GROUP BY`.

In [ ]:
sales.groupby("state_id")["d_1"].sum().sort_values(ascending=False)

### Q7. Prices — what do prices look like, and how much do stores differ?
**Why:** Summary statistics catch outliers and bad data early. Comparing stores shows whether price varies by location.

In [ ]:
print(price["sell_price"].describe())
print()
store_avg = price.groupby("store_id")["sell_price"].mean().sort_values(ascending=False)
print(store_avg)
print()
print(f"highest - lowest store average: ${store_avg.max() - store_avg.min():.2f}")

### Q8. Label alignment — what happens when two Series don't share the same labels?
**Why:** Pandas combines data by **label**, not position. Labels without a partner become `NaN`, and int columns can silently turn into floats. This is a common source of production bugs.

In [ ]:
d1_store = sales.groupby("store_id")["d_1"].sum()                                # all 10 stores
d2_ca_store = sales[sales["state_id"] == "CA"].groupby("store_id")["d_2"].sum()  # CA stores only

combined = d1_store + d2_ca_store
print(combined)
print()
print("dtypes:", d1_store.dtype, "+", d2_ca_store.dtype, "->", combined.dtype)
print("NaNs after combine:", combined.isna().sum())
print("labels with no partner:", d1_store.index.difference(d2_ca_store.index).tolist())

One fix is `fill_value=0`. Only use it when "missing" really means zero. Here it doesn't: TX and WI do have d_2 sales, they just weren't in the second Series. Filling with 0 would give a wrong total that looks right.

In [ ]:
fixed = d1_store.add(d2_ca_store, fill_value=0)
print(fixed)
print("dtype:", fixed.dtype)

### Q9. Safe updates — how do you change a subset without touching the original?
**Why:** Modifying a filtered result can silently fail or change data you didn't mean to. `.copy()` makes an independent table, and a single `.loc[mask, col] = value` changes the original directly.

In [ ]:
mask = (sales["state_id"] == "CA") & (sales["cat_id"] == "FOODS")
foods3 = sales["dept_id"] == "FOODS_3"
before = sales.loc[mask & foods3, "d_1"].sum()

# Modify an independent copy
ca_food = sales[mask].copy()
ca_food.loc[ca_food["dept_id"] == "FOODS_3", "d_1"] = 0

print("copy FOODS_3 d_1 sum :", ca_food.loc[ca_food["dept_id"] == "FOODS_3", "d_1"].sum())
print("sales FOODS_3 d_1 sum:", sales.loc[mask & foods3, "d_1"].sum())
assert sales.loc[mask & foods3, "d_1"].sum() == before
print("original unchanged: True")

**The chained-assignment trap.** This looks like it sets `d_1` to 0, but it writes to a temporary copy, so `sales` does not change. Pandas 3 warns about it.

In [ ]:
sales[mask]["d_1"] = 0
print("sales CA FOODS d_1 sum after chained assignment:", sales.loc[mask, "d_1"].sum(), "(unchanged)")

To change `sales` itself, write in one step: `sales.loc[mask, "d_1"] = 0` (not run here, so the data stays intact for the next parts).

### Part 1 findings
*(Fill in after running.)*

- Pandas version and text dtype:
- Where most of the memory goes:
- What the calendar's missing values mean:
- Largest category, and whether stores share the same catalog:
- State with the most day-1 sales:
- Price spread between stores:
- What label alignment did to the combined Series:
- How to safely change a subset vs the original: